# Credit Risk Guru — an English-named `Runtime` for personal-loan underwriting

This notebook is the English-terminology counterpart to
[`credit_risk_guru_ksetra.ipynb`](credit_risk_guru_ksetra.ipynb). Same use
case, same narrative beats, same underlying objects — but every class is
imported under its English alias from [`ear/english.py`](../ear/english.py)
instead of its Sanskrit name: `Runtime` instead of `Ksetra`, `Policy`
instead of `Dharma`, `Skill` instead of `Vidya`, and so on for all 33
classes.

These aliases are not copies. `Runtime is Ksetra` holds — an instance built
under one vocabulary is exactly an instance of the other, so a codebase can
mix names freely or migrate one class at a time.

| EAR concern | Credit risk equivalent |
| --- | --- |
| `Skill` (`Vidya`) | One scoring capability (score band, DTI band, risk grade) |
| `Persona` (`Guna`) | The credit analyst carrying out the judgement |
| `Workflow` (`Varna`) | The underwriting workflow ordering personas |
| `Process` (`Karma`) | "Personal Loan Underwriting" as a runnable process |
| `Policy` (`Dharma`) | Hard underwriting / fair-lending floors |
| `Runtime` (`Ksetra`) | The runtime battlefield running the whole pipeline |
| `Validator` (`Pariksha`) | The maker-checker layer for every pipeline stage |
| `Evidence` (`Pramana`) | The audit trail behind one decision |
| `Memory` / `Experience` / `Adaptation` | `Smriti` / `Anubhava` / `Samskara` |


## 0. Imports

Pull in the English-terminology classes — `Policy`, `Persona`, `Process`,
`Runtime`, `Validator`, `Intent`, `Skill`, `Workflow` — instead of their
Sanskrit originals.

In [1]:
import os

from ear import Persona, Policy, Process, Runtime, Validator, Intent, Skill, Workflow

print("EAR English-terminology stack ready.")


EAR English-terminology stack ready.


## 1. `Skill` — the skills a credit analyst actually uses

A `Skill` (`Vidya`) is one addressable capability. We give our persona
three small, inspectable scoring skills instead of one opaque "score the
applicant" black box — each is independently testable and swappable.

In [2]:
def credit_score_tier(score: int) -> str:
    if score >= 720:
        return "prime"
    if score >= 660:
        return "near-prime"
    if score >= 620:
        return "subprime"
    return "deep-subprime"


def dti_tier(debt_to_income_ratio: float) -> str:
    if debt_to_income_ratio <= 0.30:
        return "low"
    if debt_to_income_ratio <= 0.45:
        return "moderate"
    return "high"


_RISK_GRID = {
    ("prime", "low"): "A", ("prime", "moderate"): "A", ("prime", "high"): "B",
    ("near-prime", "low"): "B", ("near-prime", "moderate"): "B", ("near-prime", "high"): "C",
    ("subprime", "low"): "C", ("subprime", "moderate"): "C", ("subprime", "high"): "D",
    ("deep-subprime", "low"): "D", ("deep-subprime", "moderate"): "D", ("deep-subprime", "high"): "E",
}


def risk_grade(score_tier: str, dti_band: str) -> str:
    return _RISK_GRID.get((score_tier, dti_band), "E")


score_skill = Skill(name="credit_score_tier", description="Bands a FICO score", handler=credit_score_tier)
dti_skill = Skill(name="dti_tier", description="Bands a debt-to-income ratio", handler=dti_tier)
grade_skill = Skill(name="risk_grade", description="Combines score + DTI tiers into a risk grade", handler=risk_grade)

print([s.name for s in (score_skill, dti_skill, grade_skill)])


['credit_score_tier', 'dti_tier', 'risk_grade']


## 2. `Persona` — the "Credit Risk Guru" persona

A `Persona` (`Guna`) stacks `Skill`s behind a behavioural nature and
standing instructions — the persona that actually carries out the
underwriting judgement.

In [3]:
credit_risk_guru = Persona(
    name="Credit Risk Guru",
    instructions=(
        "Underwrite personal loans conservatively. Band every applicant's "
        "credit score and debt-to-income ratio, then assign a risk grade "
        "before any approval reasoning happens."
    ),
)
credit_risk_guru.add_skill(score_skill).add_skill(dti_skill).add_skill(grade_skill)

print(credit_risk_guru.name, "->", [s.name for s in credit_risk_guru.skills])


Credit Risk Guru -> ['credit_score_tier', 'dti_tier', 'risk_grade']


## 3. `Workflow` — the underwriting workflow

A `Workflow` (`Varna`) orders one or more `Persona`s into a workflow. Here
it's a single-persona workflow; a larger desk might stack a fraud-review
persona ahead of the credit-risk persona in the same `Workflow`.

In [4]:
underwriting_workflow = Workflow(name="Underwriting Workflow")
underwriting_workflow.add_persona(credit_risk_guru)

print(underwriting_workflow.name, "->", [p.name for p in underwriting_workflow.personas])


Underwriting Workflow -> ['Credit Risk Guru']


## 4. `Process` — the process

A `Process` (`Karma`) is the executable process: one or more `Workflow`s
stacked into an action a `Runtime` can discover and run.

In [5]:
loan_underwriting = Process(name="Personal Loan Underwriting")
loan_underwriting.add_workflow(underwriting_workflow)

print(loan_underwriting.name, "->", [w.name for w in loan_underwriting.workflows])


Personal Loan Underwriting -> ['Underwriting Workflow']


## 5. `Policy` — underwriting policy gates

Each `Policy` (`Dharma`) is a governance rule, safely evaluated (no
`eval`/`exec` — see `ear/_safe_eval.py`) against the `Intent`'s context.
`Governor` (`Niyamana`) checks every one of these *before* anything else in
the cycle runs, and stops the cycle outright if any is violated — these are
the bank's non-negotiable underwriting and fair-lending floors, not soft
suggestions.

In [6]:
policies = [
    Policy(name="Minimum Credit Score", rule="credit_score >= 620"),
    Policy(name="Debt-to-Income Ceiling", rule="debt_to_income_ratio <= 0.45"),
    Policy(name="No Active Defaults", rule="existing_defaults == 0"),
    Policy(name="Loan Amount Cap", rule="loan_amount <= 75000"),
]

for policy in policies:
    print(f"- {policy.name}: {policy.rule}")


- Minimum Credit Score: credit_score >= 620
- Debt-to-Income Ceiling: debt_to_income_ratio <= 0.45
- No Active Defaults: existing_defaults == 0
- Loan Amount Cap: loan_amount <= 75000


## 6. `Runtime` — assemble Credit Risk Guru's runtime

`Runtime` (`Ksetra`) is where the process, the policies, and all nineteen
pipeline stages come together. We register the process and the policies;
everything else (`Governor`, `Discoverer`, `Selector`, `Composer`,
`Scheduler`, `Validator`, `Orchestrator`, `Memory`, `Experience`,
`Adaptation`, ...) is already wired in by `Runtime`'s own `__init__`.

In [7]:
runtime = Runtime(name="Credit-Risk-Guru-Runtime")
runtime.add_process(loan_underwriting)
for policy in policies:
    runtime.add_policy(policy)

print(runtime.name)
print("Processes:", [p.name for p in runtime.processes])
print("Policies: ", [p.name for p in runtime.policies])


Credit-Risk-Guru-Runtime
Processes: ['Personal Loan Underwriting']
Policies:  ['Minimum Credit Score', 'Debt-to-Income Ceiling', 'No Active Defaults', 'Loan Amount Cap']


## 7. `ModelBinding` — optional: give the Guru a real mind

`Runtime.reason()` works with **no** LLM at all — `Reasoner` falls back to
a deterministic summary, which is what the rest of this notebook runs on.
If you want real LLM-backed reasoning instead, attach a `ModelBinding`
(`Manas`):

```python
from ear import ModelBinding

runtime.manas = ModelBinding(
    provider="openai",
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)
```

Never hardcode a key in a notebook cell or commit one to a file — read it
from the environment, as below, and rotate any key that was ever pasted
into chat.

In [8]:
if os.environ.get("OPENAI_API_KEY"):
    from ear import ModelBinding

    runtime.manas = ModelBinding(provider="openai", model="gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
    print("ModelBinding attached -- Reasoner will reason against a real LLM.")
else:
    print("No OPENAI_API_KEY in the environment -- Reasoner uses its dependency-free default reasoning.")


No OPENAI_API_KEY in the environment -- Reasoner uses its dependency-free default reasoning.


## 8. Turning an applicant into an `Intent`

In a real desk, feature computation (score banding, DTI banding, risk
grading) happens before the runtime ever reasons about the application.
`assess_applicant` invokes the Guru's own `Skill`s to compute those
features, folds them into the `Intent`'s context, then hands the whole
thing to `runtime.reason()` — so the policy gates and the explanation both
see the same banded features a human underwriter would.

In [9]:
def assess_applicant(name: str, credit_score: int, debt_to_income_ratio: float,
                      annual_income: float, loan_amount: float, existing_defaults: int = 0):
    score_tier = credit_risk_guru.get_skill("credit_score_tier").invoke(credit_score)
    dti_band = credit_risk_guru.get_skill("dti_tier").invoke(debt_to_income_ratio)
    grade = credit_risk_guru.get_skill("risk_grade").invoke(score_tier, dti_band)

    context = {
        "credit_score": credit_score,
        "debt_to_income_ratio": debt_to_income_ratio,
        "annual_income": annual_income,
        "loan_amount": loan_amount,
        "existing_defaults": existing_defaults,
        "score_tier": score_tier,
        "dti_band": dti_band,
        "risk_grade": grade,
    }
    intent = Intent(
        text=f"Underwrite a ${loan_amount:,.0f} personal loan for {name} (risk grade {grade})",
        context=context,
    )
    return intent


def show_decision(name: str, entry) -> None:
    print(f"Applicant: {name}")
    print(f"  Decision : {entry.decision}")
    print(f"  Basis    : {entry.evidence.basis}")
    print(f"  Plan     : {entry.evidence.sources['plan']}")
    print(f"  Policies checked: {entry.evidence.sources['policies_checked']}")
    print(f"  Explanation: {entry.evidence.sources['explanation']}")


## 9. Applicant A — clean approval

A prime-tier, low-DTI applicant comfortably clears every `Policy` gate.

In [10]:
intent_a = assess_applicant(
    name="Asha Rao", credit_score=760, debt_to_income_ratio=0.22,
    annual_income=95_000, loan_amount=20_000, existing_defaults=0,
)
runtime.reason(intent_a)
show_decision("Asha Rao", runtime.smriti.working[-1])


Applicant: Asha Rao
  Decision : [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $20,000 personal loan for Asha Rao (risk grade A)' across processes: Personal Loan Underwriting
  Basis    : Resolved via Bhuddi's dependency-free default
  Plan     : ['Underwriting Workflow']
  Policies checked: ['Minimum Credit Score', 'Debt-to-Income Ceiling', 'No Active Defaults', 'Loan Amount Cap']
  Explanation: Resolved via Bhuddi's dependency-free default -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $20,000 personal loan for Asha Rao (risk grade A)' across processes: Personal Loan Underwriting


## 10. Applicant B — stopped by `Governor` before anything else runs

This applicant's debt-to-income ratio breaches the 0.45 ceiling. `Governor`
(`Niyamana`) checks every `Policy` *first* — before `Discoverer` even
discovers a process — and `Runtime.reason()` raises `PermissionError`
immediately. No decision is reasoned, no `Memory` entry is written: a
policy violation never quietly becomes a yes.

In [11]:
intent_b = assess_applicant(
    name="Vikram Shah", credit_score=700, debt_to_income_ratio=0.52,
    annual_income=80_000, loan_amount=25_000, existing_defaults=0,
)
try:
    runtime.reason(intent_b)
except PermissionError as exc:
    print("Rejected before reasoning even started:")
    print(" ", exc)


Rejected before reasoning even started:
  Dharma violated: Debt-to-Income Ceiling


## 11. `Validator` — the checker layer for every maker stage

Past the `Policy` gate, four "maker" stages each produce something that the
next stage has to trust: `Discoverer` discovers candidate processes,
`Selector` selects among them, `Composer` composes their workflows into a
plan, and `Scheduler` schedules that plan. `Validator` (`Pariksha`) is the
one place all four outputs — plus `Decider`'s final decision — get checked
before being trusted, the same "four-eyes" dual control a credit desk
already applies to human underwriters. Break `Discoverer`'s output to watch
`Validator` reject it mid-cycle.

In [12]:
broken_workflow = Workflow(name="not-a-process")
original_discover = runtime.anveshana.discover
runtime.anveshana.discover = lambda *args, **kwargs: [broken_workflow]

try:
    runtime.reason(assess_applicant(
        name="Test Applicant", credit_score=700, debt_to_income_ratio=0.30,
        annual_income=70_000, loan_amount=15_000, existing_defaults=0,
    ))
except TypeError as exc:
    print("Validator rejected the malformed Discoverer output:")
    print(" ", exc)
finally:
    runtime.anveshana.discover = original_discover


Validator rejected the malformed Discoverer output:
  Anveshana candidates must contain only Karma instances


## 12. A small portfolio — building real memory and experience

One decision in isolation doesn't show `Memory`/`Experience`/`Adaptation`
compounding. Run a borderline applicant plus a small portfolio through the
same runtime so several cycles accumulate.

In [13]:
intent_c = assess_applicant(
    name="Priya Nair", credit_score=625, debt_to_income_ratio=0.40,
    annual_income=48_000, loan_amount=10_000, existing_defaults=0,
)
runtime.reason(intent_c)
show_decision("Priya Nair", runtime.smriti.working[-1])

portfolio = [
    dict(name="Daniel Ibe", credit_score=680, debt_to_income_ratio=0.35, annual_income=72_000, loan_amount=30_000),
    dict(name="Mei Lin", credit_score=740, debt_to_income_ratio=0.18, annual_income=110_000, loan_amount=50_000),
    dict(name="Carlos Diaz", credit_score=622, debt_to_income_ratio=0.44, annual_income=48_000, loan_amount=10_000),
    dict(name="Sara Kim", credit_score=705, debt_to_income_ratio=0.28, annual_income=89_000, loan_amount=40_000),
]

for applicant in portfolio:
    runtime.reason(assess_applicant(existing_defaults=0, **applicant))
    show_decision(applicant["name"], runtime.smriti.working[-1])
    print()

print(f"Total successful cycles observed by Experience: {runtime.anubhava.observations}")


Applicant: Priya Nair
  Decision : [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $10,000 personal loan for Priya Nair (risk grade C)' across processes: Personal Loan Underwriting, drawing on 1 remembered cycles
  Basis    : Resolved via Bhuddi's dependency-free default
  Plan     : ['Underwriting Workflow']
  Policies checked: ['Minimum Credit Score', 'Debt-to-Income Ceiling', 'No Active Defaults', 'Loan Amount Cap']
  Explanation: Resolved via Bhuddi's dependency-free default -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $10,000 personal loan for Priya Nair (risk grade C)' across processes: Personal Loan Underwriting, drawing on 1 remembered cycles
Applicant: Daniel Ibe
  Decision : [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $30,000 personal loan for Daniel Ibe (risk grade B)' across processes: Personal Loan Underwriting, drawing on 2 remembered cycles
  Basis    : Resolved via Bhuddi's dependency-free default
  Plan     : ['Underwriting Wo

## 13. Inspecting `Memory`, `Experience` and `Adaptation`

Six applicants have now cleared the runtime. `Adapter`'s (`Anukulana`'s)
default `adapt_every=5` means the fifth successful cycle triggered
`AdaptationBank` (`SamskaraBank`) to distill one new lesson from
`Experience`'s aggregated pattern — not on every single call, so adaptation
doesn't spam one fresh impression per decision.

In [14]:
print("=== Memory (what happened) ===")
print(runtime.smriti.context_window())

print("\n=== Experience (the pattern across repeated cycles) ===")
print(runtime.anubhava.summary())

print("\n=== Adaptation (the distilled lesson) ===")
if runtime.samskara.impressions:
    for impression in runtime.samskara.impressions:
        print(f"- {impression.name}: {impression.insight}")
else:
    print("No adaptation distilled yet.")


=== Memory (what happened) ===
Recent history:
- (2026-06-25 04:31) 'Underwrite a $20,000 personal loan for Asha Rao (risk grade A)' -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $20,000 personal loan for Asha Rao (risk grade A)' across processes: Personal Loan Underwriting [Resolved via Bhuddi's dependency-free default]
- (2026-06-25 04:31) 'Underwrite a $10,000 personal loan for Priya Nair (risk grade C)' -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $10,000 personal loan for Priya Nair (risk grade C)' across processes: Personal Loan Underwriting, drawing on 1 remembered cycles [Resolved via Bhuddi's dependency-free default]
- (2026-06-25 04:31) 'Underwrite a $30,000 personal loan for Daniel Ibe (risk grade B)' -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $30,000 personal loan for Daniel Ibe (risk grade B)' across processes: Personal Loan Underwriting, drawing on 2 remembered cycles [Resolved via Bhuddi's dependency-free default]
- (20

## 14. The full `Evidence` audit trail behind one decision

Every entry's `evidence` is an `Evidence` (`Pramana`) — separate from the
decision itself — recording which reasoning path was used, which policies
were checked, the composed plan, what `Recaller` (`Smarana`) recalled from
memory, `Explainer`'s (`Vyakhya`'s) explanation, and `Auditor`'s
(`Parishodhana`'s) audit flag. This is the artifact a credit risk
model-governance review (or an adverse-action notice under ECOA) would
actually ask to see.

In [15]:
last_entry = runtime.smriti.working[-1]
import json
print(json.dumps(
    {**last_entry.evidence.sources, "decision": last_entry.decision, "basis": last_entry.evidence.basis},
    indent=2, default=str,
))


{
  "policies_checked": [
    "Minimum Credit Score",
    "Debt-to-Income Ceiling",
    "No Active Defaults",
    "Loan Amount Cap"
  ],
  "context": {
    "credit_score": 705,
    "debt_to_income_ratio": 0.28,
    "annual_income": 89000,
    "loan_amount": 40000,
    "existing_defaults": 0,
    "score_tier": "near-prime",
    "dti_band": "low",
    "risk_grade": "B"
  },
  "plan": [
    "Underwriting Workflow"
  ],
  "recalled_memory": "Recent history:\n- (2026-06-25 04:31) 'Underwrite a $20,000 personal loan for Asha Rao (risk grade A)' -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $20,000 personal loan for Asha Rao (risk grade A)' across processes: Personal Loan Underwriting [Resolved via Bhuddi's dependency-free default]\n- (2026-06-25 04:31) 'Underwrite a $10,000 personal loan for Priya Nair (risk grade C)' -> [Credit-Risk-Guru-Runtime] resolved Sankalpa 'Underwrite a $10,000 personal loan for Priya Nair (risk grade C)' across processes: Personal Loan Underwriting,

## Why the English names change nothing underneath

- `Runtime is Ksetra`, `Policy is Dharma`, `Validator is Pariksha`, and so
  on for all 33 classes — these are aliases, not parallel implementations,
  so this notebook and `credit_risk_guru_ksetra.ipynb` are running the
  exact same pipeline, just narrated in a different vocabulary.
- **`Policy` gates** mirror hard underwriting/fair-lending floors (minimum
  score, DTI ceiling, exposure caps) — `Governor` enforces them *before*
  any reasoning runs, so a breach can never be reasoned around.
- **`Validator`'s maker-checker layer** is the same "four-eyes" /
  dual-control principle credit risk operations already require of human
  underwriters, just applied to every stage of the automated pipeline
  (`Discoverer`, `Selector`, `Composer`, `Scheduler`, `Decider`).
- **`Evidence`** gives every decision an audit trail independent of the
  decision text itself — exactly what SR 11-7-style model governance and
  adverse-action notices require: not just *what* was decided, but
  defensibly *why*.
- **`Memory` → `Experience` → `Adaptation`** keep "what happened", "the
  pattern across the portfolio", and "the adaptation drawn from it" as
  three distinct layers, instead of one model silently drifting on raw
  history — which is precisely the kind of unmonitored drift credit risk
  model governance exists to catch.

Swap the dependency-free `Reasoner` fallback for a real `ModelBinding` +
compiled DSPy program, plug in your bank's actual policy thresholds, and
this same `Runtime` is a real (if minimal) underwriting runtime — fully
inspectable, one named class per concern, at every stage, in whichever
vocabulary your team prefers.